# Lesson 07: Wiring the Endpoints

## What You're Building

In this lesson you'll assemble everything from lessons 01-06 into two complete FastAPI services: the Ingestion API (POST /ingest, GET /documents) and the Chat API (POST /chat). This is the final lesson — by the end, you'll have rebuilt both services from scratch.

The code in this notebook is functionally equivalent to what's in `services/ingestion/app/main.py` and `services/chat/app/main.py`. Compare your version to the production code when you're done.

## How This Fits in the App

This IS the app. The Ingestion API is the entry point for documents (PDF → text → chunks → embeddings → Qdrant). The Chat API is the entry point for questions (question → embedding → search → prompt → streamed answer).

In [ ]:
# Prerequisites — requires Ollama and Qdrant
import httpx
from qdrant_client import QdrantClient

OLLAMA_BASE_URL = "http://localhost:11434"
QDRANT_HOST = "localhost"
QDRANT_PORT = 6333

# Quick connectivity check
try:
    async with httpx.AsyncClient() as client:
        await client.get(f"{OLLAMA_BASE_URL}/api/tags", timeout=5.0)
    print("✓ Ollama")
except:
    print("✗ Ollama")

try:
    QdrantClient(host=QDRANT_HOST, port=QDRANT_PORT).get_collections()
    print("✓ Qdrant")
except:
    print("✗ Qdrant")

## Package Introductions

### python-multipart
Handles multipart form data — the format browsers use for file uploads. FastAPI needs this to parse `UploadFile` parameters. In Go, `multipart.Reader` does the same thing. You never call python-multipart directly — FastAPI uses it internally when you declare an `UploadFile` parameter.

### sse-starlette
Wraps an async generator into an SSE (Server-Sent Events) HTTP response. Instead of manually formatting `data: ...\n\n` lines, you yield dicts and `EventSourceResponse` handles the formatting. It's built on Starlette (the ASGI framework under FastAPI).

Key API: `EventSourceResponse(generator)` — takes an async generator, returns a streaming HTTP response.

### CORS Middleware (from FastAPI/Starlette)
CORS (Cross-Origin Resource Sharing) is the browser security mechanism that blocks requests from one origin (e.g., `localhost:3000`) to another (e.g., `localhost:8001`). The `CORSMiddleware` adds the HTTP headers that tell the browser "yes, this other origin is allowed to call me."

In Go, you'd use `rs/cors` or write the headers manually. In Express, you'd use the `cors` package. Same concept, same headers, different framework glue.

## Go/TS Comparison

| Concept | Go | Python/FastAPI |
|---------|-----|----------------|
| File upload | `r.FormFile("file")` | `file: UploadFile = File(...)` |
| Read file bytes | `io.ReadAll(file)` | `await file.read()` |
| CORS | `cors.New(cors.Options{...})` middleware | `CORSMiddleware(allow_origins=["*"])` |
| Singleton | `sync.Once` + package-level var | Module-level var + lazy init function |
| HTTP error | `http.Error(w, msg, 422)` | `raise HTTPException(status_code=422, detail=msg)` |

The **lazy singleton** pattern deserves attention: in Go, you'd use `sync.Once` to initialize a database connection exactly once. In Python, we use a module-level variable + a function that initializes it on first call. It's simpler but serves the same purpose.

## Build It

### Step 1: Define all the building blocks

Let's define all the functions from previous lessons in one place, so our endpoints can use them.

In [ ]:
# All building blocks from Lessons 02-06, assembled together

import uuid
import json
from io import BytesIO
from typing import AsyncGenerator

from PyPDF2 import PdfReader
from langchain_text_splitters import RecursiveCharacterTextSplitter
import httpx
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, PointStruct, VectorParams

# --- Config ---
OLLAMA_BASE_URL = "http://localhost:11434"
EMBEDDING_MODEL = "nomic-embed-text"
CHAT_MODEL = "mistral"
QDRANT_HOST = "localhost"
QDRANT_PORT = 6333
COLLECTION_NAME = "lesson_07_test"
CHUNK_SIZE = 1000
CHUNK_OVERLAP = 200

# --- PDF Parser (Lesson 02) ---
def extract_pages(pdf_file: BytesIO) -> list[dict]:
    try:
        content = pdf_file.read()
        if not content:
            raise ValueError("empty or invalid PDF")
        pdf_file.seek(0)
        reader = PdfReader(pdf_file)
    except Exception as e:
        if "empty or invalid" in str(e):
            raise
        raise ValueError(f"empty or invalid PDF: {e}")
    return [{"page_number": i + 1, "text": page.extract_text() or ""} for i, page in enumerate(reader.pages)]

# --- Chunker (Lesson 02) ---
def chunk_pages(pages, chunk_size=1000, chunk_overlap=200):
    splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap, length_function=len)
    chunks, index = [], 0
    for page in pages:
        text = page["text"].strip()
        if not text:
            continue
        for split in splitter.split_text(text):
            chunks.append({"text": split, "page_number": page["page_number"], "chunk_index": index})
            index += 1
    return chunks

# --- Embedder (Lesson 03) ---
async def embed_texts(texts, ollama_base_url, model):
    if not texts:
        return []
    async with httpx.AsyncClient() as client:
        response = await client.post(f"{ollama_base_url}/api/embed", json={"model": model, "input": texts}, timeout=120.0)
        response.raise_for_status()
        return response.json()["embeddings"]

# --- Store (Lesson 04) ---
class QdrantStore:
    def __init__(self, host, port, collection_name):
        self.client = QdrantClient(host=host, port=port)
        self.collection_name = collection_name
        if not self.client.collection_exists(collection_name):
            self.client.create_collection(collection_name=collection_name, vectors_config=VectorParams(size=768, distance=Distance.COSINE))

    def upsert(self, chunks, vectors, document_id, filename):
        points = [PointStruct(id=str(uuid.uuid4()), vector=v, payload={"text": c["text"], "page_number": c["page_number"], "chunk_index": c["chunk_index"], "document_id": document_id, "filename": filename}) for c, v in zip(chunks, vectors)]
        self.client.upsert(collection_name=self.collection_name, points=points)

    def list_documents(self):
        records, _ = self.client.scroll(collection_name=self.collection_name, limit=10000, with_payload=True, with_vectors=False)
        docs = {}
        for r in records:
            did = r.payload["document_id"]
            if did not in docs:
                docs[did] = {"document_id": did, "filename": r.payload["filename"], "chunks": 0}
            docs[did]["chunks"] += 1
        return list(docs.values())

# --- Prompt (Lesson 05) ---
SYSTEM_PROMPT = """You are a helpful document Q&A assistant. Answer questions based only on the provided context. If the context doesn't contain enough information to answer, say so honestly — do not make up information.\n\nWhen referencing information, mention the source file and page number."""
RAG_TEMPLATE = "Context:\n{context}\n\nQuestion: {question}\n\nAnswer based only on the context above. Cite sources (filename, page) when possible."
NO_CONTEXT_TEMPLATE = "The user asked: {question}\n\nI don't have any relevant context from uploaded documents to answer this question."

def build_rag_prompt(question, chunks):
    if not chunks:
        return NO_CONTEXT_TEMPLATE.format(question=question)
    context = "\n\n".join(f"[{c['filename']}, page {c['page_number']}]\n{c['text']}" for c in chunks)
    return RAG_TEMPLATE.format(context=context, question=question)

# --- Streaming (Lesson 06) ---
async def stream_ollama_response(prompt, model, base_url):
    async with httpx.AsyncClient() as client:
        async with client.stream("POST", f"{base_url}/api/generate", json={"model": model, "prompt": prompt, "system": SYSTEM_PROMPT, "stream": True}, timeout=300.0) as response:
            response.raise_for_status()
            async for line in response.aiter_lines():
                if line.strip():
                    data = json.loads(line)
                    if data.get("response"):
                        yield {"token": data["response"]}
                    if data.get("done"):
                        break

print("All building blocks loaded!")

### Step 2: Build the Ingestion API

This is `services/ingestion/app/main.py`. Note the patterns:
- **Lazy singleton** for the Qdrant store — created on first request, reused after
- **File validation** — check extension and size before processing
- **Error handling** — `HTTPException` for client errors (422), propagated errors from `extract_pages`

In [ ]:
from fastapi import FastAPI, File, HTTPException, UploadFile
from fastapi.middleware.cors import CORSMiddleware
from fastapi.testclient import TestClient

ingestion_app = FastAPI(title="Ingestion API")

ingestion_app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_methods=["*"], allow_headers=["*"])

# Lazy singleton — initialized on first request
_store = None

def get_store():
    global _store
    if _store is None:
        _store = QdrantStore(host=QDRANT_HOST, port=QDRANT_PORT, collection_name=COLLECTION_NAME)
    return _store

@ingestion_app.get("/health")
async def ingestion_health():
    return {"status": "ok"}

@ingestion_app.post("/ingest")
async def ingest(file: UploadFile = File(...)):
    # Validate file type
    if not file.filename or not file.filename.lower().endswith(".pdf"):
        raise HTTPException(status_code=422, detail="Only PDF files are accepted")

    # Read and validate size
    content = await file.read()
    max_bytes = 50 * 1024 * 1024  # 50MB
    if len(content) > max_bytes:
        raise HTTPException(status_code=422, detail="File exceeds 50MB limit")

    # Extract pages
    try:
        pages = extract_pages(BytesIO(content))
    except ValueError as e:
        raise HTTPException(status_code=422, detail=str(e))

    # Chunk
    chunks = chunk_pages(pages, chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP)
    if not chunks:
        raise HTTPException(status_code=422, detail="No text content found in PDF")

    # Embed
    texts = [c["text"] for c in chunks]
    vectors = await embed_texts(texts, OLLAMA_BASE_URL, EMBEDDING_MODEL)

    # Store
    document_id = str(uuid.uuid4())
    store = get_store()
    store.upsert(chunks=chunks, vectors=vectors, document_id=document_id, filename=file.filename)

    return {"status": "success", "document_id": document_id, "chunks_created": len(chunks), "filename": file.filename}

@ingestion_app.get("/documents")
async def list_documents():
    store = get_store()
    return {"documents": store.list_documents()}

print("Ingestion API defined!")

In [ ]:
# Test the ingestion API using FastAPI's test client
from fpdf import FPDF

# Create a test PDF
pdf = FPDF()
pdf.add_page()
pdf.set_font("Helvetica", size=12)
pdf.cell(0, 10, "The quarterly revenue was 2.5 million dollars.")
pdf.ln()
pdf.cell(0, 10, "Operating margins improved to 23 percent.")
test_pdf = BytesIO()
pdf.output(test_pdf)
test_pdf.seek(0)

client = TestClient(ingestion_app)

# Test health
print("Health:", client.get("/health").json())

# Test ingest
response = client.post("/ingest", files={"file": ("report.pdf", test_pdf, "application/pdf")})
print(f"Ingest: {response.json()}")

# Test documents list
print(f"Documents: {client.get('/documents').json()}")

# Test rejection of non-PDF
response = client.post("/ingest", files={"file": ("report.txt", BytesIO(b"hello"), "text/plain")})
print(f"Non-PDF rejected: {response.status_code}")

### Step 3: Build the Chat API

This is `services/chat/app/main.py`. The key new pieces:
- **SSE response** via `EventSourceResponse` — wraps our async generator
- **The full RAG flow** in `rag_query()` — embed → search → prompt → stream

In [ ]:
from pydantic import BaseModel
from sse_starlette.sse import EventSourceResponse

chat_app = FastAPI(title="Chat API")
chat_app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_methods=["*"], allow_headers=["*"])

class ChatRequest(BaseModel):
    question: str
    collection: str | None = None

async def rag_query(question, ollama_base_url, chat_model, embedding_model, qdrant_host, qdrant_port, collection_name, top_k=5):
    """Full RAG pipeline as an async generator."""
    # Embed
    vectors = await embed_texts([question], ollama_base_url, embedding_model)
    query_vector = vectors[0]

    # Retrieve
    retriever_client = QdrantClient(host=qdrant_host, port=qdrant_port)
    results = retriever_client.search(collection_name=collection_name, query_vector=query_vector, limit=top_k)
    chunks = [{"text": h.payload["text"], "page_number": h.payload["page_number"], "filename": h.payload["filename"], "document_id": h.payload["document_id"], "score": h.score} for h in results]

    # Prompt
    prompt = build_rag_prompt(question, chunks)

    # Sources
    seen = set()
    sources = []
    for c in chunks:
        key = (c["filename"], c["page_number"])
        if key not in seen:
            seen.add(key)
            sources.append({"file": c["filename"], "page": c["page_number"]})

    # Stream
    async for event in stream_ollama_response(prompt, chat_model, ollama_base_url):
        yield event
    yield {"done": True, "sources": sources}

@chat_app.get("/health")
async def chat_health():
    return {"status": "healthy"}

@chat_app.post("/chat")
async def chat(request: ChatRequest):
    async def event_generator():
        async for event in rag_query(
            question=request.question,
            ollama_base_url=OLLAMA_BASE_URL,
            chat_model=CHAT_MODEL,
            embedding_model=EMBEDDING_MODEL,
            qdrant_host=QDRANT_HOST,
            qdrant_port=QDRANT_PORT,
            collection_name=request.collection or COLLECTION_NAME,
        ):
            yield {"data": json.dumps(event)}
    return EventSourceResponse(event_generator())

print("Chat API defined!")

In [ ]:
# Test the chat API
chat_client = TestClient(chat_app)

# Health check
print("Health:", chat_client.get("/health").json())

# Chat — this will stream an SSE response
response = chat_client.post("/chat", json={"question": "What was the revenue?"})
print(f"\nStatus: {response.status_code}")
print(f"Content-Type: {response.headers.get('content-type')}")

# Parse SSE events
events = []
for line in response.text.strip().split("\n"):
    if line.startswith("data: "):
        events.append(json.loads(line[6:]))

tokens = [e["token"] for e in events if "token" in e]
done = [e for e in events if e.get("done")]

print(f"\nTokens received: {len(tokens)}")
print(f"Answer: {''.join(tokens)}")
if done:
    print(f"Sources: {done[0]['sources']}")

### Congratulations!

You've rebuilt both backend services from scratch. Compare your code to:
- `services/ingestion/app/main.py` — your Ingestion API
- `services/chat/app/main.py` — your Chat API

The production code splits things into separate files (pdf_parser.py, chunker.py, embedder.py, etc.) but the logic is identical to what you've built here.

In [ ]:
# Clean up
store = get_store()
store.client.delete_collection(COLLECTION_NAME)
_store = None
print("Cleaned up test collection")

## Check Your Understanding

1. **Why do we use a lazy singleton for the Qdrant store instead of creating it at import time?** (Hint: think about what happens during testing and when the service first starts)

2. **What does `CORSMiddleware(allow_origins=["*"])` do, and why would you tighten it for production?**

3. **Trace the full data flow for a chat request: what happens from the moment the user hits Send to when they see the first token?** (List every step: frontend → backend → Ollama → Qdrant → etc.)